# 📊 PCA Dimensionality Reduction — CC General Credit Card Dataset

**Objective:** Apply Principal Component Analysis (PCA) to reduce 17-dimensional credit card customer data, visualize structure, and identify customer segments via KMeans clustering.

| Dataset | Samples | Features | Method |
|---------|---------|----------|--------|
| CC General | 8,950 | 17 | PCA + KMeans |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings, json
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "#f8fafc",
                     "axes.grid": True, "grid.alpha": 0.3})
print("✅ Libraries loaded successfully")

## 1. Data Loading & Preprocessing

In [ ]:
df = pd.read_csv("../data/CC_GENERAL.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Drop ID, impute missing values
df.drop(columns=["CUST_ID"], inplace=True)
df["CREDIT_LIMIT"] = df["CREDIT_LIMIT"].fillna(df["CREDIT_LIMIT"].median())
df["MINIMUM_PAYMENTS"] = df["MINIMUM_PAYMENTS"].fillna(df["MINIMUM_PAYMENTS"].median())

print(f"Missing values after imputation: {df.isnull().sum().sum()}")
print(f"\nShape: {df.shape}")
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(18, 14))
axes = axes.flatten()
for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=40, color="#6366f1", alpha=0.75, edgecolor="white")
    axes[i].set_title(col, fontsize=9, fontweight="bold")
    axes[i].tick_params(labelsize=7)
for j in range(len(df.columns), len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Feature Distributions", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../models/feature_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, annot=True,
            fmt=".2f", linewidths=0.5, ax=ax, annot_kws={"size": 7})
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../models/correlation_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Standardization & PCA

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
print(f"Scaled data — mean: {X_scaled.mean():.6f}, std: {X_scaled.std():.6f}")

# Full PCA for variance analysis
pca_full = PCA()
pca_full.fit(X_scaled)
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

n90 = int(np.argmax(cumulative >= 0.90)) + 1
n95 = int(np.argmax(cumulative >= 0.95)) + 1
print(f"\nComponents needed for 90% variance: {n90}")
print(f"Components needed for 95% variance: {n95}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scree plot
colors = ["#6366f1" if i < n90 else "#d1d5db" for i in range(17)]
axes[0].bar(range(1, 18), explained * 100, color=colors, edgecolor="white")
axes[0].plot(range(1, 18), cumulative * 100, "o-", color="#f59e0b", lw=2, label="Cumulative")
ax2 = axes[0].twinx()
ax2.set_ylabel("Cumulative Variance (%)", color="#f59e0b")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Individual Variance (%)")
axes[0].set_title("Scree Plot", fontsize=13, fontweight="bold")
axes[0].legend(loc="center right")

# Cumulative variance
axes[1].plot(range(1, 18), cumulative * 100, "o-", color="#6366f1", lw=2.5)
axes[1].axhline(90, color="#10b981", ls="--", label=f"90% (n={n90})")
axes[1].axhline(95, color="#f59e0b", ls="--", label=f"95% (n={n95})")
axes[1].axvline(n90, color="#10b981", ls=":", alpha=0.7)
axes[1].axvline(n95, color="#f59e0b", ls=":", alpha=0.7)
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Explained Variance (%)")
axes[1].set_title("Cumulative Explained Variance", fontsize=13, fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../models/scree_plot.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Selected components: {n90} (explains {cumulative[n90-1]*100:.1f}% variance)")

In [ ]:
pca = PCA(n_components=n90)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA output shape: {X_pca.shape}")

variance_table = pd.DataFrame({
    "Component": [f"PC{i+1}" for i in range(n90)],
    "Explained Variance (%)": [round(v*100, 2) for v in pca.explained_variance_ratio_],
    "Cumulative (%)": [round(v*100, 2) for v in np.cumsum(pca.explained_variance_ratio_)]
})
variance_table

## 4. 2D & 3D PCA Visualization

In [ ]:
# Quick 2D scatter (coloured by density)
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.3, s=8, c="#6366f1")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)", fontsize=12)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=12)
ax.set_title("PCA — 2D Projection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../models/pca_2d.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=df.columns,
    columns=[f"PC{i+1}" for i in range(n90)]
)

# Heatmap
fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(loadings, cmap="RdBu_r", center=0, annot=True, fmt=".2f",
            linewidths=0.5, ax=ax, annot_kws={"size": 8}, vmin=-1, vmax=1)
ax.set_title("PCA Loadings — Feature Contribution per Component", fontsize=13, fontweight="bold")
ax.set_xlabel("Principal Components")
ax.set_ylabel("Features")
plt.tight_layout()
plt.savefig("../models/pca_loadings.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.15, s=6, c="#6366f1", label="Samples")

scale = 3
for i, feat in enumerate(df.columns):
    ax.annotate("", xy=(loadings.iloc[i, 0]*scale, loadings.iloc[i, 1]*scale),
                xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="#ef4444", lw=1.5))
    ax.text(loadings.iloc[i, 0]*scale*1.05, loadings.iloc[i, 1]*scale*1.05,
            feat, fontsize=7, color="#1f2937", fontweight="bold")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)", fontsize=12)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=12)
ax.set_title("PCA Biplot — Samples + Feature Loadings", fontsize=13, fontweight="bold")
ax.axhline(0, color="grey", lw=0.5, ls="--"); ax.axvline(0, color="grey", lw=0.5, ls="--")
plt.tight_layout()
plt.savefig("../models/pca_biplot.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. KMeans Clustering on PCA-Reduced Data

In [ ]:
sil_scores, inertias = {}, {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    sil_scores[k] = round(silhouette_score(X_pca, labels), 4)
    inertias[k] = km.inertia_
    print(f"k={k}: silhouette={sil_scores[k]:.4f}, inertia={inertias[k]:,.0f}")

best_k = max(sil_scores, key=sil_scores.get)
print(f"\n✅ Best k = {best_k} (silhouette = {sil_scores[best_k]:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(sil_scores.keys()), list(sil_scores.values()), "o-", color="#6366f1", lw=2.5)
axes[0].axvline(best_k, color="#ef4444", ls="--", label=f"Best k={best_k}")
axes[0].set_xlabel("Number of Clusters (k)"); axes[0].set_ylabel("Silhouette Score")
axes[0].set_title("Silhouette Score vs k", fontsize=13, fontweight="bold"); axes[0].legend()

axes[1].plot(list(inertias.keys()), list(inertias.values()), "o-", color="#f59e0b", lw=2.5)
axes[1].set_xlabel("Number of Clusters (k)"); axes[1].set_ylabel("Inertia (WCSS)")
axes[1].set_title("Elbow Method", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("../models/kmeans_selection.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km_best.fit_predict(X_pca)

palette = ["#6366f1","#10b981","#f59e0b","#ef4444","#8b5cf6","#06b6d4","#f97316","#84cc16"]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for c in range(best_k):
    mask = cluster_labels == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], s=8, alpha=0.5,
                    color=palette[c], label=f"Cluster {c}")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].set_title(f"KMeans (k={best_k}) on PCA Space", fontsize=13, fontweight="bold")
axes[0].legend()

dist = pd.Series(cluster_labels).value_counts().sort_index()
axes[1].bar([f"Cluster {i}" for i in dist.index], dist.values,
            color=palette[:best_k], edgecolor="white", linewidth=0.8)
axes[1].set_title("Cluster Distribution", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Number of Customers")
for i, v in enumerate(dist.values):
    axes[1].text(i, v + 20, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("../models/pca_clusters.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Save Results

In [ ]:
metrics = {
    "n_features_original": 17,
    "n_components_90pct": n90,
    "n_components_95pct": n95,
    "explained_variance_ratio": [round(float(v),4) for v in explained],
    "cumulative_variance": [round(float(v),4) for v in cumulative],
    "component_variance_pct": {f"PC{i+1}": round(float(pca.explained_variance_ratio_[i])*100,2)
                                for i in range(n90)},
    "kmeans_silhouette_scores": sil_scores,
    "best_k": best_k,
    "best_silhouette": sil_scores[best_k],
    "cluster_distribution": {int(k): int(v) for k,v in
                              pd.Series(cluster_labels).value_counts().sort_index().items()}
}

with open("../models/pca_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

output_df = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(n90)])
output_df["Cluster"] = cluster_labels
output_df.to_csv("../models/pca_reduced.csv", index=False)

print("✅ Saved: models/pca_metrics.json")
print("✅ Saved: models/pca_reduced.csv")
print(json.dumps(metrics, indent=2))

## 7. Summary

| Item | Value |
|------|-------|
| Original Dimensions | 17 |
| Selected Components (90% var.) | 10 |
| Variance Retained | 91.9% |
| Best k (KMeans) | 3 |
| Best Silhouette Score | 0.2586 |

**Key Findings:**
- PC1 (27.3%) & PC2 (20.3%) dominate — capturing purchase & balance behaviour
- 10 components retain 91.9% of dataset variance (41% dimensionality reduction)
- KMeans k=3 provides the best cluster separation in PCA space
- Cluster 0 (majority) = standard credit card users; Cluster 1 & 2 = high-activity segments
